# CrimeVision-QA Fine-Tuning Notebook

This notebook runs the fine-tuning pipeline on the local Windows + RTX 4090 setup using the short-path virtual environment at `C:\cvqa-venv`.

Recommended order:
1. Run the environment check cell
2. Run the stage you want to train
3. Resume later with the remaining stage cells


In [ ]:
from pathlib import Path
import subprocess
import sys

ROOT = Path.cwd()
PYTHON_EXE = Path(r"C:\cvqa-venv\Scripts\python.exe")

def run(cmd, cwd=ROOT):
    cmd = [str(x) for x in cmd]
    print('$ ' + ' '.join(cmd))
    process = subprocess.Popen(
        cmd,
        cwd=str(cwd),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end='')
    process.wait()
    if process.returncode != 0:
        raise RuntimeError(f'Command failed with exit code {process.returncode}')

print('ROOT =', ROOT)
print('PYTHON_EXE =', PYTHON_EXE)
if not PYTHON_EXE.exists():
    raise FileNotFoundError(PYTHON_EXE)


In [ ]:
run([PYTHON_EXE, '--version'])
run([PYTHON_EXE, '-c', "import torch; print('TORCH_VERSION=' + torch.__version__); print('CUDA_AVAILABLE=' + str(torch.cuda.is_available())); print('GPU_NAME=' + (torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE'))"])
run([PYTHON_EXE, '-c', "import shutil; print('FFMPEG=' + str(shutil.which('ffmpeg')))"])


## Full or partial fine-tuning

Run individual stages below. If a stage already completed, skip it and continue with the next one.


In [ ]:
run([PYTHON_EXE, 'finetune/generate_embedding_data.py', '--output', 'finetune/data/embedding_training_data.json'])
run([PYTHON_EXE, 'finetune/train_embeddings.py', '--data', 'finetune/data/embedding_training_data.json', '--output', 'finetune/output/embeddings', '--epochs', '5'])


In [ ]:
run([PYTHON_EXE, 'finetune/generate_whisper_data.py', '--frames-dir', './frames', '--output', 'finetune/data/whisper_training_data.json'])
run([PYTHON_EXE, 'finetune/train_whisper.py', '--data', 'finetune/data/whisper_training_data.json', '--output', 'finetune/output/whisper', '--epochs', '3'])


In [ ]:
run([PYTHON_EXE, 'finetune/generate_training_data.py', '--num-videos', '10', '--output', 'finetune/data/training_data.json'])
run([PYTHON_EXE, 'finetune/train_qlora.py', '--data', 'finetune/data/training_data.json', '--output', 'finetune/output/reasoner', '--epochs', '3'])


In [ ]:
run([PYTHON_EXE, 'finetune/generate_frame_data.py', '--frames-dir', './frames', '--output', 'finetune/data/frame_training_data.json'])
run([PYTHON_EXE, 'finetune/train_frame_describer.py', '--data', 'finetune/data/frame_training_data.json', '--output', 'finetune/output/frame_describer', '--epochs', '3', '--batch-size', '1'])


In [ ]:
run([PYTHON_EXE, 'evaluation/run_eval.py', '--matrix'])
